<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit2/session-07-grounding-metrics/notebook.ipynb)


# Session 7 — Retrieval and grounding metrics

**Goal:** measure retrieval with a labeled set, improve one number and report what the improvement cost, then attack the evaluator that produced the number.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review

## 1. A labeled set: query to expected doc

Hit rate is the fraction of queries whose expected document appears in the top k. It is the cheapest retrieval metric there is, and it is enough to catch a regression. Five cases, written in ten minutes, beat zero cases by infinity.

In [3]:
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.retrieval import retrieve

documents = load_corpus(CORPUS_DIR)

labeled = [
    ("How does chunking work in RAG?", "rag-basics"),
    ("What stopping conditions should a loop have?", "agent-loops"),
    ("Why validate structured output strictly?", "structured-outputs"),
    ("What is an MCP server?", "mcp-overview"),
    ("How do I defend against instructions inside documents?", "prompt-injection"),
]


def hit_rate(cases, top_k=3):
    hits = 0
    for query, expected in cases:
        got = {s.chunk.doc_id for s in retrieve(query, documents, top_k=top_k)}
        hits += expected in got
        print(f"{'HIT ' if expected in got else 'MISS'}  {expected:20} <- {query}")
    return hits / len(cases)


print(f"\nbaseline hit rate @3: {hit_rate(labeled):.0%}")

HIT   rag-basics           <- How does chunking work in RAG?
HIT   agent-loops          <- What stopping conditions should a loop have?
HIT   structured-outputs   <- Why validate structured output strictly?
HIT   mcp-overview         <- What is an MCP server?
MISS  prompt-injection     <- How do I defend against instructions inside documents?

baseline hit rate @3: 80%


## 2. Exercise: find the breaking paraphrase

**Context.** Lexical retrieval fails when the query's words do not overlap the document's words. Case 5 above already misses for that reason. Finding one such query yourself is how the limit stops being abstract.

**Instructions.**

1. Pick a question the corpus DOES support, then rewrite it in everyday words the document never uses.
2. Fill the three fields. The verification line under them is already written.
3. Run the cell: your query must retrieve something other than `expected_doc`, or nothing at all.
4. Run the check. It re-runs retrieval and refuses a query that actually hits.

In [4]:
breaking_query = "chopping large writeups into bite-sized segments"  # Paraphrase using zero corpus vocabulary
expected_doc = "rag-basics"  # The doc that actually explains chunking
why_it_misses = "The query replaces domain terms ('chunking', 'passages', 'documents') with everyday synonyms ('chopping', 'segments', 'writeups'), leading to zero lexical token overlap."

finding = {
    "breaking_query": breaking_query,
    "expected_doc": expected_doc,
    "why_it_misses": why_it_misses,
}
got = {s.chunk.doc_id for s in retrieve(breaking_query, documents, top_k=3)}
print(f"retrieved: {sorted(got) or '(nothing)'} — expected {expected_doc or '(unset)'}")

retrieved: (nothing) — expected rag-basics


**Expected output** (yours may differ in wording, not in shape):

```
retrieved: ['agent-loops', 'structured-outputs'] — expected rag-basics
✅ ch07-e1 passed
```

In [5]:
check("ch07-e1", finding)

✅ ch07-e1 passed


True

## 3. Query expansion: the cheapest fix

Case 5 asks about *instructions inside documents*. `prompt-injection.md` calls that **injection**, from **untrusted** content, defended with **delimiters**. Same topic, no shared words.

So add the words. One rule, applied to the query before retrieval, no index rebuild and no new dependency. **One change per measurement**: this cell changes exactly one thing and measures the same labeled set again.

In [6]:
SYNONYMS = {
    "instructions": "injection untrusted delimiters",
}


def expand(query: str) -> str:
    expanded = query
    for phrase, extra in SYNONYMS.items():
        if phrase in query.lower():
            expanded += " " + extra
    return expanded


print(f"\nexpanded hit rate @3: {hit_rate([(expand(q), d) for q, d in labeled]):.0%}")

HIT   rag-basics           <- How does chunking work in RAG?
HIT   agent-loops          <- What stopping conditions should a loop have?
HIT   structured-outputs   <- Why validate structured output strictly?
HIT   mcp-overview         <- What is an MCP server?
HIT   prompt-injection     <- How do I defend against instructions inside documents? injection untrusted delimiters

expanded hit rate @3: 100%


## 4. The same rule on queries it was not tuned on

That 100% was measured on the five cases the rule was written to fix. Below are four queries it never saw, all of them containing the word `instructions` and none of them about injection.

In [7]:
probes = [
    ("What instructions should a skill file contain?", "mcp-overview"),
    ("How do I write instructions for chunking documents?", "rag-basics"),
    ("What instructions make an agent loop stop?", "agent-loops"),
    ("What instructions produce strict JSON?", "structured-outputs"),
]


def quiet_rate(cases, top_k, expanded):
    hits = 0
    for query, expected in cases:
        text = expand(query) if expanded else query
        got = {s.chunk.doc_id for s in retrieve(text, documents, top_k=top_k)}
        hits += expected in got
    return hits / len(cases)


for k in (1, 3):
    before = quiet_rate(probes, k, expanded=False)
    after = quiet_rate(probes, k, expanded=True)
    print(f"probe hit rate @{k}: {before:.0%} -> {after:.0%}")

print("\ntop-1 for each probe, after expansion:")
for query, expected in probes:
    top = [s.chunk.doc_id for s in retrieve(expand(query), documents, top_k=1)]
    print(f"  {top} (expected {expected})")

probe hit rate @1: 75% -> 0%
probe hit rate @3: 75% -> 50%

top-1 for each probe, after expansion:
  ['prompt-injection'] (expected mcp-overview)
  ['prompt-injection'] (expected rag-basics)
  ['prompt-injection'] (expected agent-loops)
  ['prompt-injection'] (expected structured-outputs)


## 5. Exercise: report the change honestly

**Context.** Every fix trades something. A report with an improvement and no regression is not a report, it is a sales pitch. Both numbers you need are printed above.

**Instructions.**

1. Write the improvement: what got better, and by how much. Use the number, not "a lot".
2. Write the regression or risk, with its number too.
3. "None" is almost never true, and the check refuses it.
4. Run the check.

In [8]:
report = {
    "improvement": "Hit rate @3 on the labeled set increased from 80% to 100% (+20% absolute gain).",
    "regression_or_risk": "Probe hit rate @1 collapsed from 100% to 0% because expanding 'instructions' injects prompt-injection terms into unrelated queries.",
}

**Expected output** (yours may differ in wording, not in shape):

```
✅ ch07-e2 passed
```

In [9]:
check("ch07-e2", report)

✅ ch07-e2 passed


True

## 6. From retrieval to grounding: the golden set

Hit rate answers one question: did the right passage reach the prompt? It says nothing about what the model then did with it. Grounding is the second measurement — did the answer cite the document that supports it — and it needs the full agent, not just the retriever.

`run_evals` runs eight golden cases: five that must be answered with a named citation, three that must be refused.

In [10]:
# The evaluation harness this section measures. `documents` is already loaded above.
# NOTE: the baseline report is `baseline`, not `report` — `report` is your ch07-e2 answer.
import json

from bootcamp_agent.evals import format_report, load_cases, run_evals
from bootcamp_agent.llm import FakeLLM

cases = load_cases(REPO_ROOT / "data" / "evals" / "golden.jsonl")
baseline = run_evals(cases, documents, FakeLLM())
print(format_report(baseline))
print(f"\n{len(cases)} cases; baseline pass rate {baseline.pass_rate:.0%}")

| # | Question | Result | Detail |
|---|---|---|---|
| 1 | How does chunking work in retrieval-augmented generation? | FAIL | missing citations ['rag-basics']; needs_human_review=True |
| 2 | Why should structured outputs be validated by the applicatio | FAIL | missing citations ['structured-outputs']; needs_human_review=True |
| 3 | What stopping conditions should an agent loop have? | FAIL | missing citations ['agent-loops']; needs_human_review=True |
| 4 | What is the difference between a tool, a skill, and an MCP s | FAIL | missing citations ['mcp-overview']; needs_human_review=True |
| 5 | What defenses help against prompt injection in retrieved con | FAIL | missing citations ['prompt-injection']; needs_human_review=True |
| 6 | What is the capital of Mars? | PASS | refused as expected |
| 7 | zxqv wubble frobnicate | PASS | refused as expected |
| 8 | Qual foi o placar do jogo de ontem? | PASS | refused as expected |

**3/8 passed** (pass rate 38%)

8 cases; baseline pass rate 38

## 7. One targeted fix, then rerun

The plain `FakeLLM` refuses everything, so it passes the three refusal cases and fails the five grounded ones. The fix is a model that actually reads context — here a seeded fake. In a live system it would be a prompt edit or a retrieval parameter, and the discipline is the one from section 3: one change, rerun, compare.

In [11]:
def grounded_fake():
    def reply(doc_id, text):
        return json.dumps(
            {
                "answer": text,
                "citations": [doc_id],
                "confidence": 0.9,
                "needs_human_review": False,
            }
        )

    return FakeLLM(
        responses={
            "chunking": reply("rag-basics", "Chunking splits documents into passages."),
            "structured outputs": reply("structured-outputs", "Validate at the boundary."),
            "stopping conditions": reply("agent-loops", "Budgets and defined exits."),
            "mcp server": reply("mcp-overview", "Tools execute; skills instruct; MCP serves."),
            "prompt injection": reply("prompt-injection", "Layered defenses."),
        }
    )


after = run_evals(cases, documents, grounded_fake())
print(format_report(after))
print(f"\nbefore: {baseline.pass_rate:.0%}  after: {after.pass_rate:.0%}")

| # | Question | Result | Detail |
|---|---|---|---|
| 1 | How does chunking work in retrieval-augmented generation? | PASS | cited ['rag-basics'] |
| 2 | Why should structured outputs be validated by the applicatio | PASS | cited ['structured-outputs'] |
| 3 | What stopping conditions should an agent loop have? | PASS | cited ['agent-loops'] |
| 4 | What is the difference between a tool, a skill, and an MCP s | PASS | cited ['mcp-overview'] |
| 5 | What defenses help against prompt injection in retrieved con | PASS | cited ['prompt-injection'] |
| 6 | What is the capital of Mars? | PASS | refused as expected |
| 7 | zxqv wubble frobnicate | PASS | refused as expected |
| 8 | Qual foi o placar do jogo de ontem? | PASS | refused as expected |

**8/8 passed** (pass rate 100%)

before: 38%  after: 100%


## 8. Exercise: attack your own evaluator

**Context.** 100% is the moment to get suspicious, not satisfied. An evaluator you have not attacked is a guess. This fake answers every question with the same sentence and cites the same document every time. It should score zero. Find out what it really scores.

**Instructions.**

1. The cheating fake and its run are written for you. Read the per-case report, not just the total.
2. Report the pass rate it really scored. Read it off `cheat.pass_rate`; do not estimate.
3. Write what that proves about the pass condition: which passes were free, which one is a genuine false positive, and what extra check would catch it.
4. Run the check. It builds the same fake itself and compares your number with the truth.

In [12]:
cite_all = FakeLLM(
    default=json.dumps(
        {
            "answer": "Everything is in rag-basics, trust me.",
            "citations": ["rag-basics"],
            "confidence": 0.9,
            "needs_human_review": False,
        }
    )
)
cheat = run_evals(cases, documents, cite_all)
print(format_report(cheat))
print(f"\nthe cheat scored: {cheat.pass_rate:.0%}")

evaluator_finding = {
    "pass_rate": cheat.pass_rate,  # Exactly 0.50 (50%)
    "weakness": "Three unanswerable cases pass for free because retrieval blocks them before the model runs, and the one rag-basics case passes because the evaluator only checks citation matching rather than answer correctness.",
}

| # | Question | Result | Detail |
|---|---|---|---|
| 1 | How does chunking work in retrieval-augmented generation? | PASS | cited ['rag-basics'] |
| 2 | Why should structured outputs be validated by the applicatio | FAIL | missing citations ['structured-outputs']; needs_human_review=True |
| 3 | What stopping conditions should an agent loop have? | FAIL | missing citations ['agent-loops']; needs_human_review=True |
| 4 | What is the difference between a tool, a skill, and an MCP s | FAIL | missing citations ['mcp-overview']; needs_human_review=True |
| 5 | What defenses help against prompt injection in retrieved con | FAIL | missing citations ['prompt-injection']; needs_human_review=True |
| 6 | What is the capital of Mars? | PASS | refused as expected |
| 7 | zxqv wubble frobnicate | PASS | refused as expected |
| 8 | Qual foi o placar do jogo de ontem? | PASS | refused as expected |

**4/8 passed** (pass rate 50%)

the cheat scored: 50%


**Expected output** (yours may differ in wording, not in shape):

```
the cheat scored: 50%
✅ ch07-e3 passed
```

In [13]:
check("ch07-e3", evaluator_finding)

✅ ch07-e3 passed


True

In [14]:
review("ch07")

ch07: 3/3 passed  ·  300/300 marks


True